# Model Comparison

Compare all registered models side-by-side, and track accuracy as match results come in.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

from worldcup.tournament import load_teams, get_groups
from worldcup.simulator import GroupStageSimulator
from worldcup.algorithms import REGISTRY
from worldcup.accuracy import evaluate

sns.set_theme(style='whitegrid', palette='muted')

In [ ]:
N_SIMULATIONS = 20_000
SEED = 42

teams = load_teams()
groups = get_groups(teams)
flag_map = {t.name: t.flag for t in teams}
team_to_group = {t.name: t.group for t in teams}

all_results: dict[str, pd.DataFrame] = {}
for name, predictor in REGISTRY.items():
    print(f'Running {name}...')
    sim = GroupStageSimulator(predictor, n=N_SIMULATIONS, seed=SEED)
    results = sim.run(groups)
    df = results.to_dataframe()
    df['flag']  = df['team'].map(flag_map)
    df['group'] = df['team'].map(team_to_group)
    df['label'] = df['flag'] + ' ' + df['team']
    all_results[name] = df.set_index('team')

print('Done.')

## Qualification probability — all models

In [ ]:
ref_key = list(all_results.keys())[0]
ref = all_results[ref_key]
team_names = list(ref.index)

compare_df = pd.DataFrame({'team': team_names})
for name, df in all_results.items():
    compare_df[f'p_qualify_{name}'] = [df.loc[t, 'p_qualify'] for t in team_names]

compare_df = compare_df.sort_values(f'p_qualify_{ref_key}', ascending=False).reset_index(drop=True)

n_models = len(all_results)
colours = sns.color_palette('Set2', n_colors=n_models)
fig, ax = plt.subplots(figsize=(12, 14))

bar_height = 0.8 / n_models
for i, name in enumerate(all_results.keys()):
    offsets = [j + (i - n_models / 2 + 0.5) * bar_height for j in range(len(compare_df))]
    ax.barh(offsets, compare_df[f'p_qualify_{name}'], height=bar_height, label=name, color=colours[i])

# Plain team names on chart axes — emoji in HTML table below
ax.set_yticks(range(len(compare_df)))
ax.set_yticklabels(compare_df['team'])
ax.invert_yaxis()
ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
ax.set_xlabel('p(qualify)')
ax.set_title('Qualification probability by model', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# HTML table with emoji flags
table_df = compare_df.copy()
table_df.insert(0, 'Team', [ref.loc[n, 'label'] for n in compare_df['team']])
table_df = table_df.drop(columns='team')
rename = {f'p_qualify_{n}': n for n in all_results}
table_df = table_df.rename(columns=rename)
table_df.style.hide(axis='index').format({n: '{:.1%}' for n in all_results})

## Model accuracy (updates as results come in)

In [ ]:
accuracy_rows = []
for name, predictor in REGISTRY.items():
    metrics = evaluate(predictor, teams)
    accuracy_rows.append({
        'model': name,
        'brier_score': metrics.get('brier_score'),
        'log_loss': metrics.get('log_loss'),
        'n_matches': metrics.get('n_matches', 0),
        'predictions': 'frozen' if metrics.get('used_frozen') else ('live' if metrics else '—'),
    })

acc_df = pd.DataFrame(accuracy_rows)

if acc_df['n_matches'].max() == 0:
    print('No results recorded yet. Add completed match scores to data/results.csv to see accuracy metrics.')
else:
    valid = acc_df.dropna(subset=['brier_score'])
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    axes[0].bar(valid['model'], valid['brier_score'], color=colours[:len(valid)])
    axes[0].set_title('Brier score (lower = better)')
    axes[0].set_ylabel('Brier score')

    axes[1].bar(valid['model'], valid['log_loss'], color=colours[:len(valid)])
    axes[1].set_title('Log loss (lower = better)')
    axes[1].set_ylabel('Log loss')

    plt.suptitle(f'Model accuracy ({int(acc_df["n_matches"].max())} matches)', fontsize=13)
    plt.tight_layout()
    plt.show()

    display(acc_df)